# Plot ddrs predictions vs. USGS observations

Reads the zarr produced by `cargo run --release --bin eval` and plots:
1. A single-gauge hydrograph (predicted vs. observed).
2. The NSE distribution across all evaluation gauges.

Zarr schema (from `write_predictions_zarr`): groups `gage_ids`, `time`, `predictions`, `observations`.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

ZARR_PATH = "../output/predictions_latest.zarr"

ds = xr.open_zarr(ZARR_PATH)
ds

## Single-gauge hydrograph

In [ ]:
g = ds.gage_ids.values[0]

fig, ax = plt.subplots(figsize=(12, 4))
ds.sel(gage_ids=g).predictions.plot(ax=ax, label="ddrs")
ds.sel(gage_ids=g).observations.plot(ax=ax, label="USGS")
ax.set_title(f"Gauge {g}")
ax.set_ylabel("Discharge (m\u00b3/s)")
ax.legend()
plt.show()

## NSE distribution across gauges

In [ ]:
pred = ds.predictions.values
obs = ds.observations.values

obs_mean = np.nanmean(obs, axis=1, keepdims=True)
num = np.nansum((pred - obs) ** 2, axis=1)
den = np.nansum((obs - obs_mean) ** 2, axis=1)
nse = 1.0 - num / den
nse_finite = nse[np.isfinite(nse)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(nse_finite, bins=50, range=(-1, 1))
ax.axvline(np.median(nse_finite), color="red", linestyle="--", label=f"median = {np.median(nse_finite):.3f}")
ax.set_xlabel("NSE")
ax.set_ylabel("Gauge count")
ax.set_title(f"NSE across {len(nse_finite)} / {len(nse)} gauges (finite only)")
ax.legend()
plt.show()

print(f"mean NSE: {nse_finite.mean():.4f}")
print(f"median NSE: {np.median(nse_finite):.4f}")